<a href="https://colab.research.google.com/github/KishanGollamudi/AirLink/blob/main/tpu_llm_serving_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch_xla.core.xla_model as xm

# Query connected TPU hardware core
device = xm.xla_device()
devices = xm.get_xla_supported_devices()

print("=" * 50)
print("TPU Hardware Status: CONNECTED")
print(f"Target Execution Device: {device}")
print(f"Discovered XLA Cores   : {devices}")
print("=" * 50)

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


ModuleNotFoundError: No module named 'torch_xla'

In [2]:
import torch

# Query the native TPU device registered by the Colab runtime
device = torch.device("tpu")
print("=" * 45)
print("TPU Hardware Status: CONNECTED & READY")
print(f"Target Execution Device: {device}")
print("=" * 45)

# Run a quick matrix multiplication directly on the TPU tensor core
x = torch.randn((4, 4), device=device)
y = torch.ones((4, 4), device=device)
z = torch.matmul(x, y)

print("\nTPU Tensor Computation Result:")
print(z)

TPU Hardware Status: CONNECTED & READY
Target Execution Device: tpu

TPU Tensor Computation Result:
tensor([[ 2.1445,  2.1445,  2.1445,  2.1445],
        [-1.6797, -1.6797, -1.6797, -1.6797],
        [-1.2969, -1.2969, -1.2969, -1.2969],
        [ 1.8721,  1.8721,  1.8721,  1.8721]], device='tpu:0')


In [4]:
!pip install -q transformers accelerate sentencepiece

In [5]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("tpu")
print(f"Allocating execution context on Cloud TPU v5e ({device})...")

# 1. Download tokenizer & weights directly to TPU memory in bfloat16
model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16
).to(device)

# 2. Set up the prompt
prompt = "Explain in 3 concise bullet points why systolic arrays excel at dense GEMM operations on TPUs."
messages = [
    {"role": "system", "content": "You are an expert AI infrastructure engineer."},
    {"role": "user", "content": prompt}
]
formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_input, return_tensors="pt").to(device)

# 3. Execution & timing
print("\nCompiling graph and generating tokens on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

elapsed = time.time() - start_time
response = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(f"\n✅ Execution completed in {elapsed:.2f}s!")
print("-" * 50)
print(response)

Allocating execution context on Cloud TPU v5e (tpu)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Compiling graph and generating tokens on TPU v5e...

✅ Execution completed in 26.84s!
--------------------------------------------------
- **High Throughput**: Systolic arrays can perform multiple matrix multiplications simultaneously, significantly increasing throughput compared to traditional CPU or GPU architectures.
- **Low Power Consumption**: They use less power than CPUs and GPUs, making them suitable for resource-constrained environments like mobile devices.
- **Parallelism**: The architecture allows for parallel processing of data, which is crucial for dense GEMM operations where many elements need to be multiplied and summed together.


In [6]:
# 1. Warm inference test
bench_prompt = "Compare Kubernetes DaemonSets vs Deployments for edge observability in 2 concise bullets."
messages = [
    {"role": "system", "content": "You are a Cloud Infrastructure Engineer."},
    {"role": "user", "content": bench_prompt}
]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt").to(device)

start_time = time.time()
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

elapsed = time.time() - start_time
tokens_generated = len(output_tokens[0]) - inputs.input_ids.shape[1]
tokens_per_sec = tokens_generated / elapsed
response = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(f"⚡ Warm TPU Generation: {elapsed:.2f}s | Speed: {tokens_per_sec:.1f} tokens/sec")
print("-" * 50)
print(response)

⚡ Warm TPU Generation: 6.38s | Speed: 6.4 tokens/sec
--------------------------------------------------
- **DaemonSets** ensure that each node has exactly one instance of the application.
- **Deployments** manage the lifecycle of multiple replicas of an application across nodes, ensuring high availability and scalability.


In [7]:
# 1. Define multiple simultaneous inference requests
batch_prompts = [
    "What is the role of an Ingress Controller in Kubernetes?",
    "Explain Blue-Green deployment strategy in one sentence.",
    "Why use bfloat16 over float32 on Cloud TPUs?",
    "What is the difference between horizontal and vertical pod autoscaling?"
]

# Set padding token
tokenizer.pad_token = tokenizer.eos_token
formatted_batch = [
    tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    for p in batch_prompts
]
inputs = tokenizer(formatted_batch, return_tensors="pt", padding=True).to(device)

print(f"Executing batch size {len(batch_prompts)} concurrently on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

elapsed = time.time() - start_time
total_tokens = sum(len(out) - len(inp) for out, inp in zip(output_tokens, inputs.input_ids))
effective_tps = total_tokens / elapsed

print(f"⚡ Batched Execution: {elapsed:.2f}s | Aggregate Throughput: {effective_tps:.1f} tokens/sec")
print("=" * 60)

for idx, out in enumerate(output_tokens):
    resp = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[Prompt {idx+1}] {batch_prompts[idx]}")
    print(f"[Response] {resp.strip()}")

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Executing batch size 4 concurrently on TPU v5e...
⚡ Batched Execution: 41.58s | Aggregate Throughput: 5.8 tokens/sec

[Prompt 1] What is the role of an Ingress Controller in Kubernetes?
[Response] In,**wen. a a large Cloud. The I am curious AI designed to forQ










An Ingress Controller in Kubernetes is a component that acts as a reverse proxy and load balancer

[Prompt 2] Explain Blue-Green deployment strategy in one sentence.
[Response] **

Blue: an by Alibaba Cloud.

[Prompt 3] Why use bfloat16 over float32 on Cloud TPUs?
[Response] On Cloud TPUs, the choice between using `bfloat16` and `float32` for numerical computations can be influenced by several factors, including performance, memory usage, and precision requirements.

### Performance Considerations

1. **Memory Usage**: BFloat16 uses 4 bits

[Prompt 4] What is the difference between horizontal and vertical pod autoscaling?
[Response] **

Qwen, an Alibaba Cloud.


In [8]:
# 1. Fix decoder padding orientation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

batch_prompts = [
    "What is the role of an Ingress Controller in Kubernetes?",
    "Explain Blue-Green deployment strategy in one sentence.",
    "Why use bfloat16 over float32 on Cloud TPUs?",
    "What is the difference between horizontal and vertical pod autoscaling?"
]

formatted_batch = [
    tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    for p in batch_prompts
]

# Pad cleanly to left with fixed tensor alignment
inputs = tokenizer(
    formatted_batch,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
).to(device)

print("Running left-padded batch generation on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

elapsed = time.time() - start_time
total_tokens = sum(len(out) - len(inp) for out, inp in zip(output_tokens, inputs.input_ids))
effective_tps = total_tokens / elapsed

print(f"⚡ Clean Batch Execution: {elapsed:.2f}s | Speed: {effective_tps:.1f} tokens/sec")
print("=" * 60)

for idx, out in enumerate(output_tokens):
    # Slice off prompt input to get pure generation
    resp = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[Prompt {idx+1}] {batch_prompts[idx]}")
    print(f"[Response] {resp.strip()}")

Running left-padded batch generation on TPU v5e...
⚡ Clean Batch Execution: 6.30s | Speed: 38.1 tokens/sec

[Prompt 1] What is the role of an Ingress Controller in Kubernetes?
[Response] An Ingress Controller in Kubernetes is responsible for routing traffic to different services within a cluster based on the rules defined in an Ingress resource. It acts as a reverse proxy that forwards incoming HTTP or HTTPS requests to the appropriate backend services. The Ingress Controller can be configured to handle various types of traffic

[Prompt 2] Explain Blue-Green deployment strategy in one sentence.
[Response] Blue-Green deployment is a software deployment technique where two versions of an application are deployed side-by-side on different servers or environments, and the traffic is gradually shifted from the blue version to the green version as the green version passes through quality checks and becomes ready for production use.

[Prompt 3] Why use bfloat16 over float32 on Cloud TPUs?
[Re

In [9]:
!pip install -q fastapi uvicorn pydantic pyngrok

In [10]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import threading
import uvicorn
import time

app = FastAPI(title="Cloud TPU v5e LLM Service", version="1.0.0")

class InferenceRequest(BaseModel):
    prompt: str
    max_tokens: int = 80
    temperature: float = 0.0

class InferenceResponse(BaseModel):
    response: str
    latency_sec: float
    device: str
    tokens_generated: int

@app.get("/healthz")
def healthcheck():
    return {"status": "healthy", "accelerator": "TPU v5e-1", "backend": str(device)}

@app.post("/v1/generate", response_model=InferenceResponse)
def generate_text(req: InferenceRequest):
    try:
        t0 = time.time()

        # Prepare structured input
        messages = [{"role": "user", "content": req.prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                do_sample=(req.temperature > 0.0),
                pad_token_id=tokenizer.eos_token_id
            )

        elapsed = time.time() - t0
        new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        return InferenceResponse(
            response=decoded,
            latency_sec=round(elapsed, 3),
            device="tpu:0",
            tokens_generated=len(new_tokens)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run server on background thread inside Colab
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 FastAPI LLM Server running locally on port 8000!")

🚀 FastAPI LLM Server running locally on port 8000!


In [11]:
import requests
import json

time.sleep(2)  # Wait for uvicorn to bind

# 1. Probe healthcheck
health = requests.get("http://127.0.0.1:8000/healthz").json()
print("Health Probe:", health)

# 2. Issue an inference payload
payload = {
    "prompt": "What are three key metrics to monitor for an LLM inference service?",
    "max_tokens": 100
}
res = requests.post("http://127.0.0.1:8000/v1/generate", json=payload).json()
print("\nAPI Response:")
print(json.dumps(res, indent=2))

Health Probe: {'status': 'healthy', 'accelerator': 'TPU v5e-1', 'backend': 'tpu'}

API Response:
{
  "response": "To effectively monitor an LLM inference service, you should focus on the following three key metrics:\n\n1. **Latency**: This metric measures how long it takes for the model to process and return a response. It is crucial for understanding the performance of your LLM in terms of speed.\n\n2. **Throughput**: Throughput represents the number of requests that can be processed per unit time. High throughput indicates that your LLM can handle a large volume of queries efficiently.\n\n3. **Error Rate",
  "latency_sec": 11.87,
  "device": "tpu:0",
  "tokens_generated": 100
}


In [12]:
!pip install -q gradio

In [13]:
import gradio as gr
import time
import torch

# Ensure left-padding is active for clean generation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

def chat_tpu(message, history, max_new_tokens, temperature):
    # 1. Format the conversation using Qwen's chat template
    conversation = []
    for user_turn, bot_turn in history:
        conversation.append({"role": "user", "content": user_turn})
        if bot_turn:
            conversation.append({"role": "assistant", "content": bot_turn})
    conversation.append({"role": "user", "content": message})

    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    # 2. Tokenize and ship directly to the TPU core
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    # 3. Hardware inference
    start_time = time.time()
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=(temperature > 0.0),
            temperature=float(temperature) if temperature > 0.0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    # 4. Decode new tokens only
    new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    speed = len(new_tokens) / elapsed if elapsed > 0 else 0
    print(f"TPU Inference: {len(new_tokens)} tokens generated in {elapsed:.2f}s ({speed:.1f} tok/s)")

    return response

# Build the Web UI
demo = gr.ChatInterface(
    fn=chat_tpu,
    title="Cloud TPU v5e-1 Inference Dashboard",
    description="Interactive chatbot running on Google Cloud TPU v5e hardware via native PyTorch-XLA.",
    additional_inputs=[
        gr.Slider(minimum=20, maximum=256, value=100, step=10, label="Max New Tokens"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.1, label="Temperature (0 = Greedy)")
    ],
    theme="soft"
)

# share=True creates a public secure HTTPS URL accessible outside Colab
demo.launch(share=True, debug=False)

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'theme'

In [14]:
import gradio as gr
import time
import torch

# Ensure left-padding is active for clean generation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

def chat_tpu(message, history, max_new_tokens, temperature):
    conversation = []

    # Adapt to Gradio's history format (dict or tuple list)
    for turn in history:
        if isinstance(turn, dict):
            conversation.append({"role": turn.get("role", "user"), "content": turn.get("content", "")})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, bot_msg = turn
            if user_msg:
                conversation.append({"role": "user", "content": user_msg})
            if bot_msg:
                conversation.append({"role": "assistant", "content": bot_msg})

    conversation.append({"role": "user", "content": message})

    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    start_time = time.time()
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=(temperature > 0.0),
            temperature=float(temperature) if temperature > 0.0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    speed = len(new_tokens) / elapsed if elapsed > 0 else 0
    print(f"TPU Inference: {len(new_tokens)} tokens generated in {elapsed:.2f}s ({speed:.1f} tok/s)")

    return response

# Clean ChatInterface without invalid theme parameters
demo = gr.ChatInterface(
    fn=chat_tpu,
    title="Cloud TPU v5e-1 Inference Dashboard",
    description="Interactive chatbot running on Google Cloud TPU v5e hardware via native PyTorch-XLA.",
    additional_inputs=[
        gr.Slider(minimum=20, maximum=256, value=100, step=10, label="Max New Tokens"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.1, label="Temperature (0 = Greedy)")
    ]
)

# Launch with a public shareable tunnel URL
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2296c90bc921927a2e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
